# LeetCode 3643 — Flip Square Submatrix Vertically bằng Python

Link đề: https://leetcode.com/problems/flip-square-submatrix-vertically/ · Độ khó: **Easy** · Chủ đề: Array, Two Pointers, Matrix

Skeleton LeetCode:

```python
class Solution:
    def reverseSubmatrix(self, grid: List[List[int]], x: int, y: int, k: int) -> List[List[int]]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: trong ma trận `grid`, lấy hình vuông `k x k` có góc trên-trái là ô `(x, y)`, **lật dọc** nó (đảo thứ tự các **hàng** của hình vuông, giữ nguyên cột) và trả về ma trận mới.
2. Biết ma trận Python là **danh sách các danh sách**: `grid[r][c]`, `grid[r][a:b]` là lát cắt hàng, và hiểu **tham chiếu / sao chép** (`copy`, `deepcopy`).
3. Dùng kỹ thuật **hai con trỏ** (đầu và cuối) để hoán đổi hàng đối xứng, và kỹ thuật **gán lát cắt** để hoán đổi chỉ phần cột `y..y+k-1`.
4. Viết 4 cách: sao chép hình vuông rồi ghi đảo ngược, hoán đổi từng ô bằng hai con trỏ, hoán đổi lát cắt hàng, và dựng ma trận mới bằng công thức chỉ số; đối chiếu bằng test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: hoán đổi **cả hàng** (ra ngoài hình vuông), lặp `k` lần (hoán đổi hai lần thành không làm gì), lật **ngang** thay vì dọc, và gán lát cắt mà không giữ bản sao tạm.

## 2. Đề bài & ví dụ

Cho ma trận số nguyên `grid` kích thước `m x n` và ba số nguyên `x`, `y`, `k`. Ô `(x, y)` là **góc trên-trái** của một ma trận con **hình vuông** `k x k` (hàng `x..x+k-1`, cột `y..y+k-1`). Hãy **lật dọc** ma trận con này: **đảo ngược thứ tự các hàng** của nó (hàng đầu đổi chỗ với hàng cuối, hàng thứ hai với hàng kế cuối, ...), cột giữ nguyên. Trả về ma trận sau khi lật.

| Ví dụ | grid, x, y, k | Output | Giải thích |
|---|---|---|---|
| 1 | `[[1,2,3,4],[5,6,7,8],[9,10,11,12],[13,14,15,16]]`, `x=1, y=0, k=3` | `[[1,2,3,4],[13,14,15,8],[9,10,11,12],[5,6,7,16]]` | hình vuông `3x3` là các hàng `1..3`, cột `0..2`; hàng 1 và hàng 3 (phần cột `0..2`) đổi chỗ |
| 2 | `[[3,4,2,3],[2,3,4,2]]`, `x=0, y=2, k=2` | `[[3,4,4,2],[2,3,2,3]]` | hình vuông `2x2` ở cột `2..3`; hai hàng đổi chỗ |

Constraints: `m == grid.length`, `n == grid[i].length`; `1 <= m, n <= 50`; `1 <= grid[i][j] <= 100`; `0 <= x < m`; `0 <= y < n`; `1 <= k <= min(m - x, n - y)`.

In [1]:
grid = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
x, y, k = 1, 0, 3
print("hình vuông (các hàng của nó):")
for r in range(x, x + k):
    print(grid[r][y:y + k])

hình vuông (các hàng của nó):
[5, 6, 7]
[9, 10, 11]
[13, 14, 15]


## 3. Kiến thức nền

### 3.1. Ma trận là danh sách các danh sách

`grid[r]` là hàng `r` (một list), `grid[r][c]` là ô ở hàng `r`, cột `c`. `len(grid)` là số hàng, `len(grid[0])` là số cột. Lát cắt `grid[r][a:b]` lấy các cột `a..b-1` của hàng `r` (một list **mới**).

In [2]:
g = [[1, 2, 3], [4, 5, 6]]
print(len(g), len(g[0]), g[1][2])
print(g[1][0:2])
print(g[0][1:])

2 3 6
[4, 5]
[2, 3]


### 3.2. Hoán đổi và gán lát cắt

Hoán đổi hai biến: `a, b = b, a`. Có thể hoán đổi hai **lát cắt**: `r1[y:y+k], r2[y:y+k] = r2[y:y+k], r1[y:y+k]`. Vế phải được tính **hết trước** khi gán nên không mất dữ liệu. Nhưng nếu viết gán lần lượt `r1[y:y+k] = r2[y:y+k]` rồi `r2[y:y+k] = r1[y:y+k]` thì lần hai đã đọc dữ liệu **đã bị ghi đè**.

In [3]:
a = [1, 2, 3, 4]
b = [5, 6, 7, 8]
a[1:3], b[1:3] = b[1:3], a[1:3]
print(a, b)

c, d = [1, 2], [3, 4]
c[:] = d[:]
d[:] = c[:]            # c đã bằng d rồi!
print(c, d)

[1, 6, 7, 4] [5, 2, 3, 8]
[3, 4] [3, 4]


### 3.3. Hai con trỏ từ hai đầu

Để đảo thứ tự `k` phần tử: `lo = 0`, `hi = k - 1`; trong khi `lo < hi`: hoán đổi phần tử `lo` và `hi`, `lo += 1`, `hi -= 1`. Phần tử ở giữa (khi `k` lẻ) **tự nó** nên không cần đổi. Có đúng `k // 2` lần hoán đổi.

In [4]:
arr = [10, 20, 30, 40, 50]
lo, hi = 0, len(arr) - 1
while lo < hi:
    arr[lo], arr[hi] = arr[hi], arr[lo]
    lo += 1
    hi -= 1
print(arr, "số lần đổi =", len(arr) // 2)

[50, 40, 30, 20, 10] số lần đổi = 2


### 3.4. Tham chiếu và `deepcopy`

`b = grid` **không** sao chép; `b` và `grid` cùng trỏ một ma trận. Muốn bản sao độc lập: `[row[:] for row in grid]` hoặc `copy.deepcopy(grid)`. Bài LeetCode cho phép sửa trực tiếp `grid` rồi trả về chính nó, nhưng khi **kiểm thử** ta nên sao chép đầu vào.

In [5]:
import copy
g = [[1, 2], [3, 4]]
alias = g
clone = [row[:] for row in g]
g[0][0] = 99
print(alias[0][0], clone[0][0])
print(copy.deepcopy(g) == g, copy.deepcopy(g) is g)

99 1
True False


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1: `x=1, y=0, k=3`. Hình vuông gồm các hàng `1, 2, 3`, mỗi hàng lấy cột `0..2`:

```
hàng 1: 5  6  7        (cột 3 là 8, KHÔNG thuộc hình vuông)
hàng 2: 9 10 11
hàng 3: 13 14 15       (cột 3 là 16, KHÔNG thuộc hình vuông)
```

Lật dọc = đảo thứ tự hàng: hàng 1 <-> hàng 3, hàng 2 giữ nguyên. Hàng 1 mới là `13 14 15 | 8`, hàng 3 mới là `5 6 7 | 16`. Các cột ngoài hình vuông (`8`, `16`) **không đổi**.

Tổng quát: ô `(r, c)` trong hình vuông nhận giá trị của ô `(2x + k - 1 - r, c)` (đối xứng qua "hàng giữa" của hình vuông).

## 5. Cách 1 — Brute Force / baseline

Làm đúng nghĩa đen: (1) **sao chép** hình vuông ra một ma trận `sub` (`k` hàng, mỗi hàng `k` số); (2) **đảo** thứ tự các hàng của `sub` bằng `sub.reverse()`; (3) **ghi** `sub` trở lại đúng vị trí `(x, y)` trong `grid`.

Nút thắt: tốn thêm `O(k^2)` bộ nhớ cho `sub`; nhưng tư duy đơn giản, ít lỗi.

In [6]:
class SolutionCopy:
    def reverseSubmatrix(self, grid, x, y, k):
        sub = [grid[r][y:y + k] for r in range(x, x + k)]     # lát cắt tạo list mới
        sub.reverse()
        for i in range(k):
            for j in range(k):
                grid[x + i][y + j] = sub[i][j]
        return grid

import copy
g = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
print(SolutionCopy().reverseSubmatrix(copy.deepcopy(g), 1, 0, 3))
print(SolutionCopy().reverseSubmatrix([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2))

[[1, 2, 3, 4], [13, 14, 15, 8], [9, 10, 11, 12], [5, 6, 7, 16]]
[[3, 4, 4, 2], [2, 3, 2, 3]]


**Độ phức tạp:** Time `O(k^2)`, Space `O(k^2)`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** không cần bản sao. Đảo thứ tự `k` hàng bằng **hai con trỏ**: hàng `top = x + i` đổi với hàng `bot = x + k - 1 - i` cho `i = 0 .. k//2 - 1`. Mỗi lần đổi chỉ động vào các cột `y..y+k-1`.

**Quy tắc "nếu… thì…":**

- Chỉ lặp `i` trong `range(k // 2)` — đổi mỗi cặp **đúng một lần**.
- Nếu `k` lẻ thì hàng giữa `x + k // 2` **tự đối xứng**, không đụng tới.
- Chỉ hoán đổi cột `y..y+k-1`, các cột khác giữ nguyên.

Invariant: sau bước `i`, các hàng `x..x+i` và `x+k-1-i..x+k-1` (phần cột `y..y+k-1`) đã ở vị trí cuối cùng.

In [7]:
def flip_inplace(grid, x, y, k):
    top, bot = x, x + k - 1
    while top < bot:
        for c in range(y, y + k):
            grid[top][c], grid[bot][c] = grid[bot][c], grid[top][c]
        print(f"đổi hàng {top} <-> hàng {bot} (cột {y}..{y + k - 1})")
        top += 1
        bot -= 1
    return grid

print(flip_inplace([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 1, 0, 3))

đổi hàng 1 <-> hàng 3 (cột 0..2)
[[1, 2, 3, 4], [13, 14, 15, 8], [9, 10, 11, 12], [5, 6, 7, 16]]


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) sao chép hình vuông rồi ghi lại; (2) hai con trỏ hoán đổi từng ô tại chỗ; (3) hoán đổi **lát cắt** hàng; (4) dựng ma trận **mới** bằng công thức chỉ số.

### Cách 2 — Hai con trỏ, hoán đổi từng ô tại chỗ

Đúng như mục 6. Không dùng bộ nhớ phụ ngoài vài biến.

In [8]:
class SolutionTwoPointers:
    def reverseSubmatrix(self, grid, x, y, k):
        top, bot = x, x + k - 1
        while top < bot:
            for c in range(y, y + k):
                grid[top][c], grid[bot][c] = grid[bot][c], grid[top][c]
            top += 1
            bot -= 1
        return grid

print(SolutionTwoPointers().reverseSubmatrix([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2))
print(SolutionTwoPointers().reverseSubmatrix([[1, 2], [3, 4]], 0, 0, 1))

[[3, 4, 4, 2], [2, 3, 2, 3]]
[[1, 2], [3, 4]]


Độ phức tạp Cách 2: Time `O(k^2)` (`k/2` cặp hàng, mỗi cặp `k` ô), Space `O(1)`.

### Cách 3 — Hoán đổi lát cắt hàng

Thay vòng lặp cột bằng một phép hoán đổi lát cắt: `r1[y:y+k], r2[y:y+k] = r2[y:y+k], r1[y:y+k]`. Gọn, chạy bằng mã C bên trong Python nên nhanh hơn trong thực tế.

In [9]:
class SolutionSlice:
    def reverseSubmatrix(self, grid, x, y, k):
        for i in range(k // 2):
            r1, r2 = grid[x + i], grid[x + k - 1 - i]
            r1[y:y + k], r2[y:y + k] = r2[y:y + k], r1[y:y + k]
        return grid

print(SolutionSlice().reverseSubmatrix([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 1, 0, 3))

[[1, 2, 3, 4], [13, 14, 15, 8], [9, 10, 11, 12], [5, 6, 7, 16]]


Độ phức tạp Cách 3: Time `O(k^2)`, Space `O(k)` (lát cắt tạm của một hàng).

### Cách 4 — Dựng ma trận mới bằng công thức chỉ số

Với mỗi ô `(r, c)`: nếu nằm trong hình vuông thì lấy giá trị của `(2x + k - 1 - r, c)`, ngược lại giữ nguyên. Không sửa `grid` gốc (hữu ích khi đề cấm sửa đầu vào). Cách này dễ chứng minh đúng nhưng tốn `O(mn)` bộ nhớ.

In [10]:
class SolutionFormula:
    def reverseSubmatrix(self, grid, x, y, k):
        m, n = len(grid), len(grid[0])
        res = [row[:] for row in grid]
        for r in range(x, x + k):
            for c in range(y, y + k):
                res[r][c] = grid[2 * x + k - 1 - r][c]
        return res

g = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
print(SolutionFormula().reverseSubmatrix(g, 1, 0, 3))
print(g)      # grid gốc không đổi

[[1, 2, 3, 4], [13, 14, 15, 8], [9, 10, 11, 12], [5, 6, 7, 16]]
[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]


Độ phức tạp Cách 4: Time `O(mn)` (để sao chép) + `O(k^2)`, Space `O(mn)`.

## 8. TRACE từng bước

Trace Cách 2 với ví dụ 1 (`x=1, y=0, k=3`: `top = 1`, `bot = 3`):

| bước | top | bot | thao tác | hàng top sau | hàng bot sau |
|---|---|---|---|---|---|
| 1 | 1 | 3 | đổi cột `0..2` | `[13,14,15,8]` | `[5,6,7,16]` |
| 2 | 2 | 2 | `top < bot` sai -> dừng | (hàng giữa giữ `[9,10,11,12]`) | |

In [11]:
def trace_flip(grid, x, y, k):
    g = [row[:] for row in grid]
    top, bot = x, x + k - 1
    step = 1
    while top < bot:
        print(f"bước {step}: top={top}, bot={bot}")
        print("  trước:", g[top], g[bot])
        for c in range(y, y + k):
            g[top][c], g[bot][c] = g[bot][c], g[top][c]
        print("  sau  :", g[top], g[bot])
        top += 1
        bot -= 1
        step += 1
    print(f"dừng: top={top}, bot={bot}")
    return g

res = trace_flip([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 1, 0, 3)
print(res)
trace_flip([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2)

bước 1: top=1, bot=3
  trước: [5, 6, 7, 8] [13, 14, 15, 16]
  sau  : [13, 14, 15, 8] [5, 6, 7, 16]
dừng: top=2, bot=2
[[1, 2, 3, 4], [13, 14, 15, 8], [9, 10, 11, 12], [5, 6, 7, 16]]
bước 1: top=0, bot=1
  trước: [3, 4, 2, 3] [2, 3, 4, 2]
  sau  : [3, 4, 4, 2] [2, 3, 2, 3]
dừng: top=1, bot=0


[[3, 4, 4, 2], [2, 3, 2, 3]]

## 9. Code nộp LeetCode

In [12]:
from typing import List

class Solution:
    def reverseSubmatrix(self, grid: List[List[int]], x: int, y: int, k: int) -> List[List[int]]:
        top, bot = x, x + k - 1
        while top < bot:
            for c in range(y, y + k):
                grid[top][c], grid[bot][c] = grid[bot][c], grid[top][c]
            top += 1
            bot -= 1
        return grid

print(Solution().reverseSubmatrix([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 1, 0, 3))
print(Solution().reverseSubmatrix([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2))

[[1, 2, 3, 4], [13, 14, 15, 8], [9, 10, 11, 12], [5, 6, 7, 16]]
[[3, 4, 4, 2], [2, 3, 2, 3]]


## 10. Vì sao thuật toán đúng

- **Đảo thứ tự bằng hoán đổi đối xứng:** vị trí `i` (từ trên) và `k-1-i` là hai vị trí đối xứng; đảo thứ tự nghĩa là hoán đổi mọi cặp đối xứng. Có `k // 2` cặp; hàng giữa (nếu `k` lẻ) đã đúng chỗ.
- **Mỗi cặp đổi đúng một lần:** vòng `while top < bot` dừng khi hai con trỏ gặp/vượt nhau, nên không đổi lại (nếu đổi hai lần là trả lại ban đầu — đó là lỗi hay gặp).
- **Không động vào ngoài hình vuông:** chỉ truy cập cột `y..y+k-1` và hàng `x..x+k-1`; theo constraint `k <= min(m-x, n-y)` các chỉ số luôn hợp lệ.
- **Cách 3:** vế phải của phép gán lát cắt được tính trước, nên hoán đổi đúng.
- **Cách 4:** ô `(r, c)` có độ lệch `r - x` từ đỉnh; sau lật, độ lệch là `k - 1 - (r - x)`, tức hàng `x + k - 1 - (r - x) = 2x + k - 1 - r`; cột giữ nguyên.
- **Cách 1:** `sub.reverse()` đảo thứ tự hàng và ghi lại đúng vị trí; tương đương hoán đổi cặp.

## 11. Phân tích Time / Space Complexity

`m x n` là kích thước ma trận, `k` là cạnh hình vuông (`k <= min(m, n) <= 50`).

| Cách | Time | Space |
|---|---|---|
| 1. Sao chép hình vuông rồi ghi lại | `O(k^2)` | `O(k^2)` |
| 2. Hai con trỏ, hoán đổi từng ô | `O(k^2)` | `O(1)` |
| 3. Hoán đổi lát cắt hàng | `O(k^2)` | `O(k)` |
| 4. Dựng ma trận mới bằng công thức | `O(mn)` | `O(mn)` |

Mọi cách đều rất nhanh với `m, n <= 50`. Cách 2 tối ưu bộ nhớ; Cách 4 hữu ích khi không được sửa đầu vào.

## 12. Bộ test

Test 1: ví dụ đề và edge case (`1 <= m, n <= 50`, giá trị `1..100`, `0 <= x < m`, `0 <= y < n`, `1 <= k <= min(m-x, n-y)`). Mỗi cách nhận **bản sao** của đầu vào.

In [13]:
import copy
impls = [SolutionCopy(), SolutionTwoPointers(), SolutionSlice(), SolutionFormula(), Solution()]

def oracle(grid, x, y, k):
    m, n = len(grid), len(grid[0])
    out = []
    for r in range(m):
        row = []
        for c in range(n):
            if x <= r < x + k and y <= c < y + k:
                row.append(grid[x + (x + k - 1 - r)][c])
            else:
                row.append(grid[r][c])
        out.append(row)
    return out

def valid(grid, x, y, k):
    m, n = len(grid), len(grid[0])
    return (1 <= m <= 50 and 1 <= n <= 50 and all(len(r) == n for r in grid)
            and all(1 <= v <= 100 for r in grid for v in r)
            and 0 <= x < m and 0 <= y < n and 1 <= k <= min(m - x, n - y))

tests = [
    (([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 1, 0, 3),
     [[1, 2, 3, 4], [13, 14, 15, 8], [9, 10, 11, 12], [5, 6, 7, 16]]),
    (([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2), [[3, 4, 4, 2], [2, 3, 2, 3]]),
    (([[7]], 0, 0, 1), [[7]]),                                         # 1x1
    (([[1, 2], [3, 4]], 0, 0, 1), [[1, 2], [3, 4]]),                   # k = 1: không đổi
    (([[1, 2], [3, 4]], 0, 0, 2), [[3, 4], [1, 2]]),                   # toàn ma trận
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 0, 0, 3), [[7, 8, 9], [4, 5, 6], [1, 2, 3]]),   # k lẻ
    (([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 1, 1, 2), [[1, 2, 3], [4, 8, 9], [7, 5, 6]]),   # góc dưới-phải
    (([[1, 2, 3, 4, 5]], 0, 2, 1), [[1, 2, 3, 4, 5]]),                 # ma trận 1 hàng
    (([[1], [2], [3], [4]], 1, 0, 1), [[1], [2], [3], [4]]),           # 1 cột
]
for args, exp in tests:
    assert valid(*args), args
    assert oracle(*args) == exp, (args, oracle(*args), exp)
    for im in impls:
        g, x, y, k = args
        assert im.reverseSubmatrix(copy.deepcopy(g), x, y, k) == exp, (type(im).__name__, args)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 9


Test 2: ngẫu nhiên có seed, đúng constraints (kích thước tới `50 x 50`).

In [14]:
import random
random.seed(3643)

for t in range(2000):
    m, n = random.randint(1, 50), random.randint(1, 50)
    grid = [[random.randint(1, 100) for _ in range(n)] for _ in range(m)]
    x, y = random.randrange(m), random.randrange(n)
    k = random.randint(1, min(m - x, n - y))
    assert valid(grid, x, y, k)
    exp = oracle(grid, x, y, k)
    for im in impls:
        got = im.reverseSubmatrix(copy.deepcopy(grid), x, y, k)
        assert got == exp, (type(im).__name__, m, n, x, y, k)
print("2000 test ngẫu nhiên OK")

2000 test ngẫu nhiên OK


Test 3: tính chất — lật hai lần trả về ma trận ban đầu; tổng và tập giá trị (đa tập) không đổi; ô ngoài hình vuông không đổi; `k = 1` không làm gì; Cách 4 không sửa đầu vào.

In [15]:
random.seed(643)
from collections import Counter
for _ in range(800):
    m, n = random.randint(1, 20), random.randint(1, 20)
    grid = [[random.randint(1, 100) for _ in range(n)] for _ in range(m)]
    x, y = random.randrange(m), random.randrange(n)
    k = random.randint(1, min(m - x, n - y))
    once = Solution().reverseSubmatrix(copy.deepcopy(grid), x, y, k)
    twice = Solution().reverseSubmatrix(copy.deepcopy(once), x, y, k)
    assert twice == grid
    assert Counter(v for r in once for v in r) == Counter(v for r in grid for v in r)
    for r in range(m):
        for c in range(n):
            if not (x <= r < x + k and y <= c < y + k):
                assert once[r][c] == grid[r][c]
    before = copy.deepcopy(grid)
    SolutionFormula().reverseSubmatrix(grid, x, y, k)
    assert grid == before
print("tính chất (lật hai lần, đa tập, ngoài hình vuông, không sửa input ở Cách 4): OK")

tính chất (lật hai lần, đa tập, ngoài hình vuông, không sửa input ở Cách 4): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: hoán đổi **cả hàng** (ra ngoài hình vuông)

In [16]:
def wrong_swap_rows(grid, x, y, k):
    top, bot = x, x + k - 1
    while top < bot:
        grid[top], grid[bot] = grid[bot], grid[top]            # SAI: đổi cả hàng, kể cả cột ngoài hình vuông
        top += 1
        bot -= 1
    return grid

args = ([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]], 1, 0, 3)
print("sai :", wrong_swap_rows(copy.deepcopy(args[0]), *args[1:]))
print("đúng:", Solution().reverseSubmatrix(copy.deepcopy(args[0]), *args[1:]))
assert wrong_swap_rows(copy.deepcopy(args[0]), *args[1:]) != Solution().reverseSubmatrix(copy.deepcopy(args[0]), *args[1:])

sai : [[1, 2, 3, 4], [13, 14, 15, 16], [9, 10, 11, 12], [5, 6, 7, 8]]
đúng: [[1, 2, 3, 4], [13, 14, 15, 8], [9, 10, 11, 12], [5, 6, 7, 16]]


Vì sao sai: cột `3` (các giá trị `8` và `16`) nằm **ngoài** hình vuông nhưng bị đổi theo. **Sửa:** chỉ đổi các cột `y..y+k-1`.

### Lỗi 2: lặp `k` lần (đổi mỗi cặp hai lần)

In [17]:
def wrong_twice(grid, x, y, k):
    for i in range(k):                                         # SAI: phải là range(k // 2)
        r1, r2 = grid[x + i], grid[x + k - 1 - i]
        r1[y:y + k], r2[y:y + k] = r2[y:y + k], r1[y:y + k]
    return grid

args = ([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2)
print("sai :", wrong_twice(copy.deepcopy(args[0]), *args[1:]))
print("đúng:", Solution().reverseSubmatrix(copy.deepcopy(args[0]), *args[1:]))
assert wrong_twice(copy.deepcopy(args[0]), *args[1:]) != Solution().reverseSubmatrix(copy.deepcopy(args[0]), *args[1:])

sai : [[3, 4, 2, 3], [2, 3, 4, 2]]
đúng: [[3, 4, 4, 2], [2, 3, 2, 3]]


Vì sao sai: với `k = 2`, `i = 0` đổi hai hàng, `i = 1` đổi lại -> ma trận như ban đầu. **Sửa:** chỉ `k // 2` lần.

### Lỗi 3: lật **ngang** thay vì lật dọc

In [18]:
def wrong_horizontal(grid, x, y, k):
    for r in range(x, x + k):
        grid[r][y:y + k] = grid[r][y:y + k][::-1]               # SAI: đảo cột trong mỗi hàng
    return grid

args = ([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2)
print("sai :", wrong_horizontal(copy.deepcopy(args[0]), *args[1:]))
print("đúng:", Solution().reverseSubmatrix(copy.deepcopy(args[0]), *args[1:]))
assert wrong_horizontal(copy.deepcopy(args[0]), *args[1:]) != Solution().reverseSubmatrix(copy.deepcopy(args[0]), *args[1:])

sai : [[3, 4, 3, 2], [2, 3, 2, 4]]
đúng: [[3, 4, 4, 2], [2, 3, 2, 3]]


Vì sao sai: "lật dọc" (vertically) đảo thứ tự **hàng**; đảo cột trong từng hàng là lật ngang. Ví dụ 2 cho `[[3,4,3,2],[2,3,2,4]]` thay vì `[[3,4,4,2],[2,3,2,3]]`.

### Lỗi 4: gán lát cắt lần lượt mà không giữ bản tạm

In [19]:
def wrong_overwrite(grid, x, y, k):
    for i in range(k // 2):
        r1, r2 = grid[x + i], grid[x + k - 1 - i]
        r1[y:y + k] = r2[y:y + k]                                # SAI: r1 bị ghi đè, mất dữ liệu cũ
        r2[y:y + k] = r1[y:y + k]
    return grid

args = ([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2)
print("sai :", wrong_overwrite(copy.deepcopy(args[0]), *args[1:]))
print("đúng:", Solution().reverseSubmatrix(copy.deepcopy(args[0]), *args[1:]))
assert wrong_overwrite(copy.deepcopy(args[0]), *args[1:]) != Solution().reverseSubmatrix(copy.deepcopy(args[0]), *args[1:])

sai : [[3, 4, 4, 2], [2, 3, 4, 2]]
đúng: [[3, 4, 4, 2], [2, 3, 2, 3]]


Vì sao sai: sau `r1[...] = r2[...]` thì dữ liệu cũ của `r1` mất; dòng tiếp theo chép lại chính dữ liệu đó, cả hai hàng trở nên giống nhau. **Sửa:** hoán đổi cùng lúc `a, b = b, a` hoặc dùng biến tạm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Lật / đảo thứ tự" -> **hai con trỏ** từ hai đầu, `k // 2` lần.
- "Lật dọc" = đảo thứ tự hàng (Vertical flip); "lật ngang" = đảo thứ tự cột. Dễ nhầm, hãy kiểm tra với ví dụ nhỏ.
- Công thức đối xứng: vị trí `r` trong đoạn `[x, x+k-1]` đối xứng với `2x + k - 1 - r`.
- Hoán đổi lát cắt: `a[l:r], b[l:r] = b[l:r], a[l:r]` — Python xử lý đúng nhờ tính vế phải trước.
- Khi cần giữ nguyên đầu vào: sao chép bằng `[row[:] for row in grid]` (không dùng `grid.copy()` vì chỉ sao chép nông).
- Kiểm tra biên bằng ma trận `1 x 1`, `k = 1`, và `k = min(m, n)`.

## 15. Biến thể của bài

1. **Lật ngang** hình vuông (đảo thứ tự cột trong mỗi hàng).
2. **Xoay 90 độ** hình vuông `k x k` (bài `48. Rotate Image` áp dụng cho một vùng con).
3. Lật dọc một **hình chữ nhật** `h x w` thay vì hình vuông.
4. Thực hiện **nhiều** thao tác lật trên cùng ma trận, theo danh sách truy vấn `(x, y, k)`.
5. Đếm số lần phải lật để ma trận về trạng thái ban đầu (mỗi lật lẻ/chẵn).

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sao chép + ghi lại | trích hình vuông, đảo, ghi | `O(k^2)` | `O(k^2)` | dễ hiểu, baseline |
| 2. Hai con trỏ từng ô | hoán đổi từng ô đối xứng | `O(k^2)` | `O(1)` | **nên nộp**, tại chỗ |
| 3. Hoán đổi lát cắt hàng | gán lát cắt cùng lúc | `O(k^2)` | `O(k)` | ngắn gọn, Pythonic |
| 4. Ma trận mới theo công thức | `res[r][c] = grid[2x+k-1-r][c]` | `O(mn)` | `O(mn)` | không được sửa đầu vào |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().reverseSubmatrix([[1, 2], [3, 4]], 0, 0, 2)` là gì?

**Bài 2 (tính tay):** `grid = [[1,2,3],[4,5,6],[7,8,9]]`, `x = 0`, `y = 1`, `k = 2`. Hình vuông là gì, kết quả là gì?

**Bài 3 (điền TODO):** hoàn thành `flip_horizontal(grid, x, y, k)` lật **ngang** hình vuông `k x k` tại chỗ.

**Bài 4 (tự viết lại):** viết `flip_rect_vertical(grid, x, y, h, w)` lật dọc hình chữ nhật `h` hàng, `w` cột.

**Bài 5 (mini challenge):** viết `flip_many(grid, queries)` áp dụng lần lượt nhiều truy vấn `(x, y, k)`; sau đó chứng minh (bằng assert) rằng lặp cùng một truy vấn hai lần trả về ma trận ban đầu.

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().reverseSubmatrix([[1, 2], [3, 4]], 0, 0, 2))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def flip_horizontal(grid, x, y, k):
    # TODO: với mỗi hàng r trong x..x+k-1: đảo thứ tự lát cắt grid[r][y:y+k]
    return grid

[[3, 4], [1, 2]]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** vẽ hình vuông ra giấy; lật dọc = hàng trên xuống dưới và ngược lại; cột không đổi.

**Mức 2 (vừa):** Bài 3: `grid[r][y:y+k] = grid[r][y:y+k][::-1]` cho từng hàng. Bài 4: giống bài gốc nhưng đổi `h // 2` cặp hàng, mỗi cặp `w` cột. Bài 5: gọi lần lượt; lật hai lần cùng tham số là identity.

**Mức 3 (gần đáp án):** Bài 1: hai hàng đổi chỗ -> `[[3, 4], [1, 2]]`. Bài 2: hình vuông cột `1..2`: hàng 0 `[2, 3]`, hàng 1 `[5, 6]`; sau lật: `[[1, 5, 6], [4, 2, 3], [7, 8, 9]]`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1
assert Solution().reverseSubmatrix([[1, 2], [3, 4]], 0, 0, 2) == [[3, 4], [1, 2]]

# Bài 2
assert Solution().reverseSubmatrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 0, 1, 2) == [[1, 5, 6], [4, 2, 3], [7, 8, 9]]

# Bài 3
def flip_horizontal(grid, x, y, k):
    for r in range(x, x + k):
        grid[r][y:y + k] = grid[r][y:y + k][::-1]
    return grid
assert flip_horizontal([[3, 4, 2, 3], [2, 3, 4, 2]], 0, 2, 2) == [[3, 4, 3, 2], [2, 3, 2, 4]]
assert flip_horizontal([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 0, 0, 3) == [[3, 2, 1], [6, 5, 4], [9, 8, 7]]

# Bài 4
def flip_rect_vertical(grid, x, y, h, w):
    top, bot = x, x + h - 1
    while top < bot:
        grid[top][y:y + w], grid[bot][y:y + w] = grid[bot][y:y + w], grid[top][y:y + w]
        top += 1
        bot -= 1
    return grid
assert flip_rect_vertical([[1, 2, 3], [4, 5, 6], [7, 8, 9]], 0, 0, 3, 2) == [[7, 8, 3], [4, 5, 6], [1, 2, 9]]
assert flip_rect_vertical([[1, 2, 3], [4, 5, 6]], 0, 1, 2, 2) == [[1, 5, 6], [4, 2, 3]]

# Bài 5
def flip_many(grid, queries):
    for x, y, k in queries:
        Solution().reverseSubmatrix(grid, x, y, k)
    return grid
base = [[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
assert flip_many(copy.deepcopy(base), [(1, 0, 3), (1, 0, 3)]) == base
assert flip_many(copy.deepcopy(base), [(0, 0, 2)]) == [[5, 6, 3, 4], [1, 2, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "đảo thứ tự hàng của một vùng con":**

```python
top, bot = x, x + k - 1
while top < bot:
    for c in range(y, y + k):                       # chỉ cột của vùng con
        grid[top][c], grid[bot][c] = grid[bot][c], grid[top][c]
    top += 1
    bot -= 1
return grid
```

**Checklist trước khi submit:**

- [ ] Chỉ hoán đổi các cột `y..y+k-1` (không đổi cả hàng)?
- [ ] Đổi mỗi cặp hàng **một lần** (`top < bot`, hoặc `range(k // 2)`)?
- [ ] Lật **dọc** = đảo thứ tự hàng, không phải đảo cột?
- [ ] Hoán đổi cùng lúc (`a, b = b, a`), không ghi đè rồi đọc lại?
- [ ] Đã thử `k = 1`, `k = min(m, n)`, `k` lẻ và `k` chẵn, ma trận 1 hàng / 1 cột?
- [ ] Chữ ký đúng `reverseSubmatrix(self, grid, x, y, k) -> List[List[int]]` và trả về ma trận?